# 1.2 Athena catalog: raw_pamap2 (space-delimited) and curated_pamap2 (Parquet)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3
from src.pamap2.config import load_config, s3_uri
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
sts = session.client("sts")
print("Caller:", sts.get_caller_identity()["Arn"])
from src.pamap2 import schema
import io, time, pandas as pd
athena = session.client("athena")
s3 = session.client("s3")
def run(sql, timeout=600):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=cfg["athena_workgroup"],
        QueryExecutionContext={"Database": cfg["glue_database"]})["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        qe = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        st = qe["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"Athena query exceeded {timeout}s: {sql[:200]}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql}")
    if qe.get("StatementType") != "DML":
        return None
    out = qe["ResultConfiguration"]["OutputLocation"]
    b, k = out[len("s3://"):].split("/", 1)
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))

In [ ]:
def col_ddl(cols):
    return ",\n  ".join(f"`{c}` {'int' if c == 'activity_id' else 'double'}" for c in cols)
raw_ddl = f"""CREATE EXTERNAL TABLE IF NOT EXISTS {cfg['raw_table']} (
  {col_ddl(schema.COLUMNS)}
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ' '
STORED AS TEXTFILE
LOCATION '{s3_uri(cfg, 'raw', 'pamap2/')}'"""
cur_ddl = f"""CREATE EXTERNAL TABLE IF NOT EXISTS {cfg['curated_table']} (
  {col_ddl(schema.CURATED_COLUMNS)}
)
PARTITIONED BY (session string, subject_id int)
STORED AS PARQUET
LOCATION '{s3_uri(cfg, 'curated', 'pamap2/')}'"""
run(raw_ddl); run(cur_ddl)
run(f"MSCK REPAIR TABLE {cfg['curated_table']}")
print("tables ready")

In [ ]:
print(run(f"SELECT subject_id, COUNT(*) AS n FROM {cfg['curated_table']} GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT activity_id, COUNT(*) AS n FROM {cfg['curated_table']} GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT COUNT(*) AS raw_rows FROM {cfg['raw_table']}"))